In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
PROJECT_DIR = Path(
    r"C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection"
)

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "CWRU"
)

WINDOW_DIR = PROCESSED_DIR / "windows"

RESULTS_DIR = PROJECT_DIR / "results"

INT8_MODEL_PATH = (
    RESULTS_DIR
    / "quantized"
    / "vaac_tiny_int8.tflite"
)

TEST_METADATA = (
    WINDOW_DIR
    / "test_metadata.csv"
)

print("Project directory:")
print(PROJECT_DIR)

print("\nTest metadata:")
print(TEST_METADATA)

print("\nINT8 model:")
print(INT8_MODEL_PATH)

print("\nTest metadata exists:", TEST_METADATA.exists())
print("INT8 model exists:", INT8_MODEL_PATH.exists())

Project directory:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection

Test metadata:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test_metadata.csv

INT8 model:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\vaac_tiny_int8.tflite

Test metadata exists: True
INT8 model exists: True


In [3]:
test_df = pd.read_csv(TEST_METADATA)

print("Test metadata loaded.")
print("Number of test windows:", len(test_df))

print("\nColumns:")
print(test_df.columns.tolist())

print("\nTest class distribution:")
print(test_df["class"].value_counts())

Test metadata loaded.
Number of test windows: 136

Columns:
['recording_id', 'source_file', 'signal_id', 'class', 'window_id', 'start_sample', 'end_sample', 'label']

Test class distribution:
class
Healthy       79
Ball          19
Inner Race    19
Outer Race    19
Name: count, dtype: int64


In [4]:
CLASS_NAMES = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race"
]

CLASS_TO_LABEL = {
    "Healthy": 0,
    "Ball": 1,
    "Inner Race": 2,
    "Outer Race": 3
}

LABEL_TO_CLASS = {
    0: "Healthy",
    1: "Ball",
    2: "Inner Race",
    3: "Outer Race"
}

print("CLASS MAPPING")
print("=" * 40)

for name, label in CLASS_TO_LABEL.items():
    print(f"{name:15s} -> {label}")

CLASS MAPPING
Healthy         -> 0
Ball            -> 1
Inner Race      -> 2
Outer Race      -> 3


In [5]:
interpreter = tf.lite.Interpreter(
    model_path=str(INT8_MODEL_PATH)
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INT8 TFLite interpreter initialized.")
print()
print("Input details:")
print(input_details)

print()
print("Output details:")
print(output_details)

INT8 TFLite interpreter initialized.

Input details:
[{'name': 'serving_default_vibration_input:0', 'index': 0, 'shape': array([    1, 12000,     1], dtype=int32), 'shape_signature': array([   -1, 12000,     1], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.027604417875409126, -5), 'quantization_parameters': {'scales': array([0.02760442], dtype=float32), 'zero_points': array([-5], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]

Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 48, 'shape': array([1, 4], dtype=int32), 'shape_signature': array([-1,  4], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.00390625, -128), 'quantization_parameters': {'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [6]:
input_shape = input_details[0]["shape"]
input_dtype = input_details[0]["dtype"]

output_shape = output_details[0]["shape"]
output_dtype = output_details[0]["dtype"]

print("=" * 60)
print("INT8 MODEL INTERFACE")
print("=" * 60)

print("Input shape     :", input_shape)
print("Input datatype  :", input_dtype)

print("Output shape    :", output_shape)
print("Output datatype :", output_dtype)

INT8 MODEL INTERFACE
Input shape     : [    1 12000     1]
Input datatype  : <class 'numpy.int8'>
Output shape    : [1 4]
Output datatype : <class 'numpy.int8'>


In [7]:
input_scale, input_zero_point = (
    input_details[0]["quantization"]
)

output_scale, output_zero_point = (
    output_details[0]["quantization"]
)

print("=" * 60)
print("QUANTIZATION PARAMETERS")
print("=" * 60)

print("Input scale      :", input_scale)
print("Input zero-point :", input_zero_point)

print()

print("Output scale     :", output_scale)
print("Output zero-point:", output_zero_point)

QUANTIZATION PARAMETERS
Input scale      : 0.027604417875409126
Input zero-point : -5

Output scale     : 0.00390625
Output zero-point: -128


In [8]:
def get_window_path(row):

    recording_id = str(row["recording_id"])
    window_id = int(row["window_id"])

    filename = (
        f"{recording_id}_window_{window_id:04d}.npy"
    )

    candidate_dirs = [

        # Original split
        WINDOW_DIR / "train",
        WINDOW_DIR / "validation",
        WINDOW_DIR / "test",

        # Corrected split
        PROCESSED_DIR / "windows_corrected" / "train",
        PROCESSED_DIR / "windows_corrected" / "validation",
        PROCESSED_DIR / "windows_corrected" / "test",

        # Corrected root
        PROCESSED_DIR / "windows_corrected",

        # Windows root
        WINDOW_DIR
    ]

    for directory in candidate_dirs:

        candidate = directory / filename

        if candidate.exists():
            return candidate

    # Last-resort recursive search
    matches = list(
        PROCESSED_DIR.rglob(filename)
    )

    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(
        f"Could not locate window file: {filename}"
    )

In [9]:
sample_row = test_df.iloc[0]

sample_path = get_window_path(sample_row)

print("Recording ID:", sample_row["recording_id"])
print("Window ID   :", sample_row["window_id"])

print("\nResolved path:")
print(sample_path)

print("\nFile exists:", sample_path.exists())

Recording ID: B007_3_X121
Window ID   : 0

Resolved path:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train\B007_3_X121_window_0000.npy

File exists: True


In [10]:
sample_signal = np.load(sample_path)

print("Signal shape :", sample_signal.shape)
print("Signal dtype :", sample_signal.dtype)
print("Minimum      :", sample_signal.min())
print("Maximum      :", sample_signal.max())
print("Mean         :", sample_signal.mean())
print("Std          :", sample_signal.std())

Signal shape : (12000,)
Signal dtype : float64
Minimum      : -0.6096190419161677
Maximum      : 0.6239133333333333
Mean         : 0.003428179876912841
Std          : 0.1547666237361117


In [11]:
def standardize_signal(signal):

    signal = signal.astype(np.float32)

    mean = np.mean(signal)

    std = np.std(signal)

    standardized = (
        signal - mean
    ) / (
        std + 1e-8
    )

    return standardized.astype(np.float32)

In [12]:
def quantize_input(signal):

    quantized = (
        np.round(
            signal / input_scale
        )
        + input_zero_point
    )

    quantized = np.clip(
        quantized,
        -128,
        127
    )

    return quantized.astype(np.int8)

In [13]:
standardized_sample = standardize_signal(
    sample_signal
)

standardized_sample = standardized_sample.reshape(
    1,
    12000,
    1
)

quantized_sample = quantize_input(
    standardized_sample
)

print("=" * 60)
print("PREPROCESSING / QUANTIZATION CHECK")
print("=" * 60)

print("Standardized shape:",
      standardized_sample.shape)

print("Standardized dtype:",
      standardized_sample.dtype)

print("Standardized min:",
      standardized_sample.min())

print("Standardized max:",
      standardized_sample.max())

print()

print("Quantized shape:",
      quantized_sample.shape)

print("Quantized dtype:",
      quantized_sample.dtype)

print("Quantized min:",
      quantized_sample.min())

print("Quantized max:",
      quantized_sample.max())

PREPROCESSING / QUANTIZATION CHECK
Standardized shape: (1, 12000, 1)
Standardized dtype: float32
Standardized min: -3.961107
Standardized max: 4.0091662

Quantized shape: (1, 12000, 1)
Quantized dtype: int8
Quantized min: -128
Quantized max: 127


In [14]:
def run_int8_inference(signal):

    # ------------------------------------------------
    # 1. Convert signal to FP32
    # ------------------------------------------------

    signal = signal.astype(np.float32)

    # ------------------------------------------------
    # 2. Apply training-time standardization
    # ------------------------------------------------

    signal = standardize_signal(signal)

    # ------------------------------------------------
    # 3. Reshape for model
    # ------------------------------------------------

    signal = signal.reshape(
        1,
        12000,
        1
    )

    # ------------------------------------------------
    # 4. Quantize FP32 -> INT8
    # ------------------------------------------------

    quantized_signal = quantize_input(
        signal
    )

    # ------------------------------------------------
    # 5. Send INT8 tensor to TFLite
    # ------------------------------------------------

    interpreter.set_tensor(
        input_details[0]["index"],
        quantized_signal
    )

    # ------------------------------------------------
    # 6. Execute model
    # ------------------------------------------------

    interpreter.invoke()

    # ------------------------------------------------
    # 7. Retrieve INT8 output
    # ------------------------------------------------

    raw_output = interpreter.get_tensor(
        output_details[0]["index"]
    )

    raw_output = raw_output.astype(
        np.int32
    )

    # ------------------------------------------------
    # 8. Dequantize output
    # ------------------------------------------------

    output = (
        raw_output - output_zero_point
    ) * output_scale

    output = output.flatten()

    # ------------------------------------------------
    # 9. Determine predicted class
    # ------------------------------------------------

    predicted_label = int(
        np.argmax(output)
    )

    # ------------------------------------------------
    # 10. Convert scores into normalized values
    # ------------------------------------------------

    shifted = (
        output - np.max(output)
    )

    exp_output = np.exp(
        shifted
    )

    probabilities = (
        exp_output /
        np.sum(exp_output)
    )

    confidence = float(
        probabilities[predicted_label]
    )

    return (
        predicted_label,
        confidence,
        probabilities,
        raw_output
    )

In [15]:
(
    predicted_label,
    confidence,
    probabilities,
    raw_output
) = run_int8_inference(
    sample_signal
)

true_label = CLASS_TO_LABEL[
    sample_row["class"]
]

print("=" * 60)
print("SINGLE-WINDOW INT8 INFERENCE")
print("=" * 60)

print("True class:",
      sample_row["class"])

print(
    "True label:",
    true_label
)

print(
    "Predicted label:",
    predicted_label
)

print(
    "Predicted class:",
    LABEL_TO_CLASS[
        predicted_label
    ]
)

print(
    "Confidence:",
    confidence
)

print()

print("Probabilities:")

for i, class_name in enumerate(
    CLASS_NAMES
):

    print(
        f"{class_name:15s}: "
        f"{probabilities[i]:.6f}"
    )

print()

print("Raw INT8 output:")
print(raw_output)

SINGLE-WINDOW INT8 INFERENCE
True class: Ball
True label: 1
Predicted label: 1
Predicted class: Ball
Confidence: 0.45583509994595817

Probabilities:
Healthy        : 0.177812
Ball           : 0.455835
Inner Race     : 0.188542
Outer Race     : 0.177812

Raw INT8 output:
[[-128  113 -113 -128]]


In [20]:
results = []

print(
    "Starting INT8 inference..."
)

for index, row in test_df.iterrows():

    path = get_window_path(row)

    signal = np.load(path)

    true_label = CLASS_TO_LABEL[
        row["class"]
    ]

    (
        predicted_label,
        confidence,
        probabilities,
        raw_output
    ) = run_int8_inference(
        signal
    )

    results.append({

        "recording_id":
            row["recording_id"],

        "source_file":
            row["source_file"],

        "signal_id":
            row["signal_id"],

        "class":
            row["class"],

        "window_id":
            row["window_id"],

        "true_label":
            true_label,

        "predicted_label":
            predicted_label,

        "predicted_class":
            LABEL_TO_CLASS[
                predicted_label
            ],

        "confidence":
            confidence,

        "prob_Healthy":
            probabilities[0],

        "prob_Ball":
            probabilities[1],

        "prob_Inner_Race":
            probabilities[2],

        "prob_Outer_Race":
            probabilities[3]
    })

    if (
        (index + 1) % 20 == 0
        or index == len(test_df) - 1
    ):

        print(
            f"Processed "
            f"{index + 1}/"
            f"{len(test_df)}"
        )

int8_results_df = pd.DataFrame(
    results
)

print()
print(
    "INT8 inference completed."
)

print(
    "Total windows:",
    len(int8_results_df)
)

Starting INT8 inference...
Processed 20/136
Processed 40/136
Processed 60/136
Processed 80/136
Processed 100/136
Processed 120/136
Processed 136/136

INT8 inference completed.
Total windows: 136


In [21]:
print("=" * 60)
print("INT8 PREDICTED CLASS DISTRIBUTION")
print("=" * 60)

print(
    int8_results_df[
        "predicted_class"
    ].value_counts()
)

INT8 PREDICTED CLASS DISTRIBUTION
predicted_class
Healthy       79
Ball          19
Inner Race    19
Outer Race    19
Name: count, dtype: int64


In [22]:
print(
    int8_results_df[
        "predicted_label"
    ].value_counts()
    .sort_index()
)

predicted_label
0    79
1    19
2    19
3    19
Name: count, dtype: int64


In [23]:
y_true_int8 = (
    int8_results_df["true_label"]
)

y_pred_int8 = (
    int8_results_df["predicted_label"]
)

int8_accuracy = accuracy_score(
    y_true_int8,
    y_pred_int8
)

int8_precision = precision_score(
    y_true_int8,
    y_pred_int8,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0
)

int8_recall = recall_score(
    y_true_int8,
    y_pred_int8,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0
)

int8_f1 = f1_score(
    y_true_int8,
    y_pred_int8,
    labels=[0, 1, 2, 3],
    average="weighted",
    zero_division=0
)

print("=" * 60)
print("STEP 229 — INT8 TFLITE TEST METRICS")
print("=" * 60)

print(
    "Test windows:",
    len(int8_results_df)
)

print()

print(
    "Accuracy  :",
    f"{int8_accuracy:.4f}"
)

print(
    "Precision :",
    f"{int8_precision:.4f}"
)

print(
    "Recall    :",
    f"{int8_recall:.4f}"
)

print(
    "F1-score  :",
    f"{int8_f1:.4f}"
)

STEP 229 — INT8 TFLITE TEST METRICS
Test windows: 136

Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1-score  : 1.0000


In [24]:
correct_predictions = int(
    np.sum(
        y_true_int8.values
        ==
        y_pred_int8.values
    )
)

incorrect_predictions = (
    len(int8_results_df)
    -
    correct_predictions
)

print()
print(
    "Correct predictions  :",
    correct_predictions
)

print(
    "Incorrect predictions:",
    incorrect_predictions
)


Correct predictions  : 136
Incorrect predictions: 0


In [25]:
cm_int8 = confusion_matrix(
    y_true_int8,
    y_pred_int8,
    labels=[0, 1, 2, 3]
)

print()
print("=" * 60)
print("INT8 CONFUSION MATRIX")
print("=" * 60)

print(cm_int8)


INT8 CONFUSION MATRIX
[[79  0  0  0]
 [ 0 19  0  0]
 [ 0  0 19  0]
 [ 0  0  0 19]]


In [26]:
print()
print("=" * 60)
print("INT8 CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_true_int8,
        y_pred_int8,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)


INT8 CLASSIFICATION REPORT
              precision    recall  f1-score   support

     Healthy       1.00      1.00      1.00        79
        Ball       1.00      1.00      1.00        19
  Inner Race       1.00      1.00      1.00        19
  Outer Race       1.00      1.00      1.00        19

    accuracy                           1.00       136
   macro avg       1.00      1.00      1.00       136
weighted avg       1.00      1.00      1.00       136



In [27]:
confidence_values = (
    int8_results_df[
        "confidence"
    ].values
)

print("=" * 60)
print("INT8 CONFIDENCE STATISTICS")
print("=" * 60)

print(
    "Mean   :",
    confidence_values.mean()
)

print(
    "Minimum:",
    confidence_values.min()
)

print(
    "Maximum:",
    confidence_values.max()
)

INT8 CONFIDENCE STATISTICS
Mean   : 0.4689001713545726
Minimum: 0.4261482576430023
Maximum: 0.4740677101156684


In [28]:
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INT8_RESULTS_PATH = (
    RESULTS_DIR
    / "vaac_tiny_int8_test_predictions.csv"
)

int8_results_df.to_csv(
    INT8_RESULTS_PATH,
    index=False
)

print(
    "Saved INT8 predictions to:"
)

print(
    INT8_RESULTS_PATH
)

Saved INT8 predictions to:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny_int8_test_predictions.csv


In [29]:
print()
print("=" * 70)
print("STEP 229 — INT8 TFLITE FULL TEST-SET INFERENCE")
print("=" * 70)

print(
    "Model                 : VAAC-Tiny INT8"
)

print(
    "Model file            :",
    INT8_MODEL_PATH.name
)

print(
    "Test windows          :",
    len(int8_results_df)
)

print(
    "Input shape           :",
    tuple(input_shape)
)

print(
    "Input datatype        :",
    input_dtype
)

print(
    "Output shape          :",
    tuple(output_shape)
)

print(
    "Output datatype       :",
    output_dtype
)

print()

print(
    "Accuracy              :",
    f"{int8_accuracy:.4f}"
)

print(
    "Precision             :",
    f"{int8_precision:.4f}"
)

print(
    "Recall                :",
    f"{int8_recall:.4f}"
)

print(
    "F1-score              :",
    f"{int8_f1:.4f}"
)

print()

print(
    "Correct predictions   :",
    correct_predictions
)

print(
    "Incorrect predictions :",
    incorrect_predictions
)

print()

print(
    "Results file:"
)

print(
    INT8_RESULTS_PATH
)

print()

print(
    "Step 229 completed."
)


STEP 229 — INT8 TFLITE FULL TEST-SET INFERENCE
Model                 : VAAC-Tiny INT8
Model file            : vaac_tiny_int8.tflite
Test windows          : 136
Input shape           : (np.int32(1), np.int32(12000), np.int32(1))
Input datatype        : <class 'numpy.int8'>
Output shape          : (np.int32(1), np.int32(4))
Output datatype       : <class 'numpy.int8'>

Accuracy              : 1.0000
Precision             : 1.0000
Recall                : 1.0000
F1-score              : 1.0000

Correct predictions   : 136
Incorrect predictions : 0

Results file:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\vaac_tiny_int8_test_predictions.csv

Step 229 completed.
